In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import sranodec as anom


from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler

import seaborn as sns
import matplotlib.pyplot as plt

# Data Preprocessing

In [8]:
from pathlib import Path

# Paths
ROOT = Path("/Users/mac/Desktop/paper/code/MTAD-GAT")
DATA_DIR = ROOT / "data" / "DATA"
SAVE_DIR = ROOT / "data" / "DATA" / "data_pre"
SPLIT_DIR = SAVE_DIR / "split_info"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)

# 97 subjects: 70 train/val, 27 test (fixed seed for reproducibility)
np.random.seed(42)
all_files = sorted([f.name for f in DATA_DIR.glob("*.csv") if f.suffix == ".csv"])
np.random.shuffle(all_files)
train_val_ids = [Path(f).stem for f in all_files[:70]]
test_ids = [Path(f).stem for f in all_files[70:]]

# Save split info
with open(SPLIT_DIR / "train_val_subjects.txt", "w") as f:
    f.write("\n".join(train_val_ids))
with open(SPLIT_DIR / "test_subjects.txt", "w") as f:
    f.write("\n".join(test_ids))

print(f"Train/Val subjects: {len(train_val_ids)}")
print(f"Test subjects: {len(test_ids)}")

In [9]:
def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Same handling for train and test: linear interpolation."""
    return df.interpolate(method="linear").ffill().bfill()


def preprocess_subject(subject_id: str, is_train: bool) -> pd.DataFrame:
    """
    Preprocess one subject:
    1. Per-subject normalization (MinMax)
    2. Missing value handling (same for train/test)
    3. SR clean (train only)
    """
    fpath = DATA_DIR / f"{subject_id}.csv"
    df = pd.read_csv(fpath, header=None)
    
    # 1. Per-subject MinMax normalization
    scaler = MinMaxScaler()
    arr = scaler.fit_transform(df.values)
    df = pd.DataFrame(arr, columns=df.columns)
    
    # 2. Missing value handling (same for train and test)
    df = handle_missing_values(df)
    
    # 3. SR clean: train only
    if is_train:
        for col in df.columns:
            spec = anom.Silency(24, 24, 100)
            score = spec.generate_anomaly_score(df[col].values)
            threshold = np.percentile(score, 99)
            df.loc[score > threshold, col] = np.nan
        df = handle_missing_values(df)
    
    return df

In [10]:
# Process train/val subjects
for subject_id in tqdm(train_val_ids, desc="Train/Val"):
    df = preprocess_subject(subject_id, is_train=True)
    df.to_csv(SAVE_DIR / f"train_{subject_id}.csv", index=False)

# Process test subjects
for subject_id in tqdm(test_ids, desc="Test"):
    df = preprocess_subject(subject_id, is_train=False)
    df.to_csv(SAVE_DIR / f"test_{subject_id}.csv", index=False)

print(f"Done. Train: {len(train_val_ids)}, Test: {len(test_ids)}")

################### Start 117021.csv###################
################### Finish ###################
################### Start 107018.csv###################
################### Finish ###################
################### Start 118528.csv###################
################### Finish ###################
################### Start 110613.csv###################
################### Finish ###################
################### Start 110411.csv###################
################### Finish ###################
################### Start 119025.csv###################
################### Finish ###################
################### Start 100206.csv###################
################### Finish ###################
################### Start 118932.csv###################
################### Finish ###################
################### Start 115219.csv###################
################### Finish ###################
################### Start 107422.csv###################
#################